# Position-angle vetting on DP2 deep-coadd cutouts

Previous name `dp2_pa_vetting_v8.ipynb`.

Reads a CSV of targets, fetches a single-band deep-coadd cutout (i, falling back to r, then z), shows it inverted, and lets you click two points to measure the position angle of the line joining them (counter-clockwise from +x, in [0, 180) deg). The annotated PNG is saved and the measurement is written to a results CSV.

Cutout code adapted from `dp2_deep_coadd_RGB_cutout.ipynb`. Run on the [RSP](https://data.lsst.cloud).

**Usage**
- Move the mouse over the image to see the pixel position, raw pixel value and RA/Dec (line above the image).
- Left-click two points (a third click starts a new pair). Turn off pan/zoom before clicking.
- *stretch* / *vmax %*: display only; *linear* or *unsharp* help separate blended cores.
- *Save*: writes the PNG and the CSV row. Choose the CSV columns with `OUTPUT_COLUMNS` in the config cell.

Input CSV: one row per target with name, ra, dec (deg); any other columns are ignored. Column names are matched case-insensitively; edit `COLS` if yours differ. Target names are kept exactly as written in the CSV.

In [ ]:
%matplotlib widget

from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth
from pyvo.dal.adhoc import SodaQuery

import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import ipywidgets as widgets
from IPython.display import display
from astropy.visualization import AsinhStretch, LinearStretch, LogStretch, ImageNormalize
from astropy import units as u
from scipy.ndimage import gaussian_filter

In [ ]:
# ---------------- configuration ----------------
TARGET_CSV = 'targets.csv'

# map logical field -> column name in TARGET_CSV
COLS = dict(name='name', ra='ra', dec='dec')

OUT_DIR = 'pa_vetting_output'
RESULT_CSV = os.path.join(OUT_DIR, 'pa_results.csv')

# columns written to RESULT_CSV, in this order (None = all). Delete the ones you don't need.
#   x1, y1, x2, y2     : clicked pixel positions
#   ra1, dec1, ...     : clicked positions on the sky (deg)
#   pa_pix_deg         : PA counter-clockwise from +x, [0, 180)
#   pa_sky_deg         : PA east of north, [0, 180)
# Keep 'name': it is used to overwrite a re-saved target.
# If you change this list, start a new RESULT_CSV (old and new rows would be merged).
OUTPUT_COLUMNS = ['name', 'ra', 'dec', 'band',
                  'x1', 'y1', 'x2', 'y2',
                  'ra1', 'dec1', 'ra2', 'dec2',
                  'sep_arcsec', 'pa_pix_deg', 'pa_sky_deg']
FLOAT_FORMAT = None         # e.g. '%.6f' to fix the number of decimals

BANDS = ('i', 'r', 'z')     # tried in this order
RADIUS_DEG = 0.0012         # cutout radius (0.0012 deg = 4.32 arcsec)
PIX_SCALE = 0.2             # arcsec / pix
SCALEBAR_PIX = 5            # 5 pix = 1 arcsec
CIRCLE_RADIUS_PIX = 2.0     # radius of the marker circles

# display defaults (changeable with the widgets)
DEFAULT_STRETCH = 'linear'  # 'asinh', 'linear', 'log', 'unsharp'
DEFAULT_PCT_HI = 99.95
ASINH_A = 0.1
UNSHARP_SIGMA = 2.0         # pix, Gaussian subtracted in 'unsharp' mode

# figure
FIG_SIZE = (8.0, 10.1)      # inches
FONT = dict(title=18, info=15, pa=15, coord=13, axis=14, tick=12, annot=14)
COMPASS_LEN_PIX = 5.0       # length of N/E arrows

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
discovery = RSPDiscovery("dp2")
sia_client = discovery.get_sia_client()

In [ ]:
# ---------------- cutouts (from the original notebook) ----------------
def get_dl_result(band, ra, dec, radius):

    circle = (ra, dec, radius)

    results = sia_client.search(pos=circle, calib_level=3,
                                dpsubtype='lsst.deep_coadd')
    print("Num of sia_client search result: ", len(results))

    band_arr = results['lsst_band']

    try:
        index = int(np.where(band_arr == band)[-1][-1])
    except Exception:
        print(f'Band {band} does not exist!')
        return None

    dl_result = discovery.get_datalink_results(results[index])

    print(f"Datalink status: {dl_result.status}.")

    return dl_result


def get_cutout(band, ra, dec, radius=RADIUS_DEG):

    dl_result = get_dl_result(band, ra, dec, radius)

    if dl_result is None:
        return None

    sq = SodaQuery.from_resource(dl_result,
                                 dl_result.get_adhocservice_by_id("cutout-sync"),
                                 session=get_pyvo_auth())

    sq.circle = (ra * u.deg, dec * u.deg, radius * u.deg)

    cutout_bytes = sq.execute_stream().read()
    sq.raise_if_error()

    cutout = read_archive(io.BytesIO(cutout_bytes))

    return cutout


def image_array(cutout):
    """Pixel array; works for Image and MaskedImage-like objects."""
    img = cutout.image if hasattr(cutout, 'image') else cutout
    return np.asarray(img.array, dtype=float)


def image_unit(cutout):
    try:
        img = cutout.image if hasattr(cutout, 'image') else cutout
        return '' if img.unit is None else str(img.unit)
    except Exception:
        return ''


def get_cutout_fallback(ra, dec, bands=BANDS, radius=RADIUS_DEG):
    """Return (cutout, band) for the first band with usable data, else (None, None)."""
    for band in bands:
        try:
            cutout = get_cutout(band, ra, dec, radius)
        except Exception as e:
            print(f'{band}: cutout failed ({e})')
            continue
        if cutout is None:
            continue
        data = image_array(cutout)
        if data.size == 0 or not np.any(np.isfinite(data) & (data != 0)):
            print(f'{band}: cutout is empty')
            continue
        return cutout, band
    return None, None

In [ ]:
# ---------------- helpers ----------------
def get_wcs(cutout):
    """Astropy-API WCS whose pixel (0, 0) is the first array pixel."""
    for attr in ('astropy_wcs', 'fits_wcs'):
        try:
            w = getattr(cutout, attr)
            if w is not None:
                return w
        except Exception:
            pass
    return None


def pix2sky(wcs, x, y):
    if wcs is None:
        return None
    try:
        return wcs.pixel_to_world(x, y)
    except Exception:
        return None


def sky_pa_sep(wcs, x1, y1, x2, y2):
    """(PA east of north mod 180, separation arcsec, (ra1, dec1, ra2, dec2))."""
    s1, s2 = pix2sky(wcs, x1, y1), pix2sky(wcs, x2, y2)
    if s1 is None or s2 is None:
        return np.nan, np.hypot(x2 - x1, y2 - y1) * PIX_SCALE, (np.nan,) * 4
    return (float(s1.position_angle(s2).deg % 180.0), float(s1.separation(s2).arcsec),
            (s1.ra.deg, s1.dec.deg, s2.ra.deg, s2.dec.deg))


def draw_compass(ax, wcs, nx, ny, length=COMPASS_LEN_PIX, color='tab:blue'):
    """N and E arrows in the upper-left corner, directions from the WCS."""
    x0, y0 = length + 4, ny - length - 5
    dirs = {'N': (0.0, 1.0), 'E': (-1.0, 0.0)}      # fallback: north up, east left
    c0 = pix2sky(wcs, x0, y0)
    if c0 is not None:
        try:
            d = {}
            for lab, (dlon, dlat) in (('N', (0, 1)), ('E', (1, 0))):
                c1 = c0.spherical_offsets_by(dlon * u.arcsec, dlat * u.arcsec)
                px, py = wcs.world_to_pixel(c1)
                v = np.array([float(px) - x0, float(py) - y0])
                d[lab] = v / np.hypot(*v)
            dirs = d
        except Exception as e:
            print(f'Compass from WCS failed ({e}); assuming north up, east left.')
    for lab, (vx, vy) in dirs.items():
        ax.annotate('', xy=(x0 + length * vx, y0 + length * vy), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='-|>', color=color, lw=2, mutation_scale=18))
        ax.text(x0 + (length + 2.2) * vx, y0 + (length + 2.2) * vy, lab, color=color,
                fontsize=FONT['annot'], weight='bold', ha='center', va='center')


def coord_str(k, x, y, ra, dec):
    s = f'{k}:  x, y = {x:.2f}, {y:.2f}'
    if np.isfinite(ra):
        s += f'      RA, Dec = {ra:.6f}, {dec:+.6f}'
    return s


def display_image(data, mode='asinh', pct_hi=99.8):
    img = np.array(data, float)
    if mode == 'unsharp':                      # high-pass: removes wings, keeps cores
        img = img - gaussian_filter(np.nan_to_num(img), UNSHARP_SIGMA)
    finite = img[np.isfinite(img)]
    lo = 0.0 if mode == 'unsharp' else np.nanmedian(finite)
    hi = np.percentile(finite, pct_hi)
    if hi <= lo:
        hi = lo + 1.0
    stretch = {'asinh': AsinhStretch(ASINH_A), 'linear': LinearStretch(),
               'log': LogStretch(1000), 'unsharp': LinearStretch()}[mode]
    return img, ImageNormalize(vmin=lo, vmax=hi, stretch=stretch, clip=True)


def pa_deg(x1, y1, x2, y2):
    return float(np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180.0)


def file_name(s):
    """Target name as written; only '/' (a path separator) is replaced."""
    return str(s).replace('/', '_')


def toolbar_active(canvas):
    """True if pan/zoom is active (then clicks should be ignored)."""
    tb = getattr(canvas, 'toolbar', None)
    if tb is None:
        return False
    mode = getattr(tb, 'mode', None)
    if mode is not None and str(mode) != '':
        return True
    return bool(getattr(tb, '_current_action', ''))

In [ ]:
# ---------------- interactive vetter ----------------
class PAVetter:

    def __init__(self, targets):
        self.t = targets.reset_index(drop=True)
        self.cache = {}
        self.pts, self.artists = [], []
        self.result = None
        self.data = self.wcs = None
        self.unit = ''

        with plt.ioff():
            self.fig, self.ax = plt.subplots(figsize=FIG_SIZE)
        H = FIG_SIZE[1]
        self.fig.subplots_adjust(left=0.12, right=0.97, top=1 - 1.2 / H, bottom=1.8 / H)
        self.fig.canvas.header_visible = False
        self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        self.fig.canvas.mpl_connect('motion_notify_event', self.on_move)

        names = self.t[COLS['name']]
        self.dd = widgets.Dropdown(options=[(f'{k}: {n}', k) for k, n in enumerate(names)],
                                   value=0, description='Target')
        self.dd.observe(lambda ch: self.load(ch['new']), names='value')
        b_prev = widgets.Button(description='◀ Prev')
        b_next = widgets.Button(description='Next ▶')
        b_reset = widgets.Button(description='Reset points')
        b_save = widgets.Button(description='Save', button_style='success')
        b_prev.on_click(lambda _: self.step(-1))
        b_next.on_click(lambda _: self.step(+1))
        b_reset.on_click(lambda _: self.reset_points(redraw=True))
        b_save.on_click(self.save)
        self.advance = widgets.Checkbox(value=True, description='next after save', indent=False)
        self.mode = widgets.Dropdown(options=['asinh', 'linear', 'log', 'unsharp'],
                                     value=DEFAULT_STRETCH, description='stretch')
        self.pct = widgets.FloatSlider(value=DEFAULT_PCT_HI, min=99.0, max=100.0, step=0.05,
                                       description='vmax %', readout_format='.2f')
        for w in (self.mode, self.pct):
            w.observe(lambda ch: self.update_display(), names='value')
        self.status = widgets.HTML()
        self.cursor = widgets.HTML(value='&nbsp;')
        self.out = widgets.Output()

        self.ui = widgets.VBox([
            widgets.HBox([self.dd, b_prev, b_next]),
            widgets.HBox([b_reset, b_save, self.advance]),
            widgets.HBox([self.mode, self.pct]),
            self.status, self.cursor, self.fig.canvas, self.out])

        self.load(0)

    # ---- navigation ----
    @property
    def row(self):
        return self.t.loc[self.i]

    def step(self, d):
        j = self.dd.value + d
        if 0 <= j < len(self.t):
            self.dd.value = j          # triggers load()

    def load(self, i):
        self.i = i
        row = self.row
        with self.out:
            self.out.clear_output()
            if i not in self.cache:
                print(f'Fetching {row[COLS["name"]]} ...')
                self.cache[i] = get_cutout_fallback(float(row[COLS['ra']]),
                                                    float(row[COLS['dec']]))
        self.cutout, self.band = self.cache[i]
        self.draw_base()
        self.status.value = (f'<b>{i + 1}/{len(self.t)}</b> &nbsp; band: {self.band}'
                             if self.cutout is not None
                             else f'<b>{i + 1}/{len(self.t)}</b> &nbsp; <span style="color:red">no i/r/z cutout</span>')

    # ---- drawing ----
    def draw_base(self):
        row, ax, fig = self.row, self.ax, self.fig
        ax.clear()
        for txt in list(fig.texts):
            txt.remove()
        self.pts, self.artists, self.result = [], [], None

        ra, dec = float(row[COLS['ra']]), float(row[COLS['dec']])
        H = fig.get_figheight()
        fig.text(0.5, 1 - 0.45 / H, f"{row[COLS['name']]}",
                 ha='center', va='center', fontsize=FONT['title'], weight='bold')
        fig.text(0.5, 1 - 0.85 / H,
                 f"RA, Dec = {ra:.6f}, {dec:+.6f}      band: {self.band or '--'}",
                 ha='center', va='center', fontsize=FONT['info'])

        # bottom block, stacked upward from the bottom edge
        specs = [('clk2', 'coord', 'tab:red', 'normal'),
                 ('clk1', 'coord', 'tab:red', 'normal'),
                 ('clkpa', 'pa', 'tab:red', 'bold')]
        self.txt = {}
        y = 0.15
        for key, f, col, wt in specs:
            self.txt[key] = fig.text(0.5, y / H, '', ha='center', va='bottom',
                                     fontsize=FONT[f], color=col, weight=wt)
            y += FONT[f] * 1.45 / 72 + (0.1 if f == 'pa' else 0.0)
        self.txt['clkpa'].set_text('PA: click two points')

        if self.cutout is None:
            self.data = self.wcs = None
            self.txt['clkpa'].set_text('')
            ax.text(0.5, 0.5, 'No i / r / z cutout', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_axis_off()
            fig.canvas.draw_idle()
            return

        ax.set_axis_on()
        self.data = image_array(self.cutout)
        self.wcs = get_wcs(self.cutout)
        self.unit = image_unit(self.cutout)
        ny, nx = self.data.shape

        cmap = plt.get_cmap('gray_r').copy()
        cmap.set_bad('lightgray')
        img, norm = display_image(self.data, self.mode.value, self.pct.value)
        self.im = ax.imshow(img, origin='lower', cmap=cmap, norm=norm,
                            interpolation='nearest')
        ax.set_xlabel('x [pix]', fontsize=FONT['axis'])
        ax.set_ylabel('y [pix]', fontsize=FONT['axis'])
        ax.tick_params(labelsize=FONT['tick'])

        # scale bar: SCALEBAR_PIX pixels, lower right
        xb = nx - 3 - SCALEBAR_PIX
        yb = 3
        ax.plot([xb, xb + SCALEBAR_PIX], [yb, yb], color='tab:blue', lw=3,
                solid_capstyle='butt')
        ax.text(xb + SCALEBAR_PIX / 2, yb + 1, f'{SCALEBAR_PIX * PIX_SCALE:g}"',
                ha='center', va='bottom', color='tab:blue', fontsize=FONT['annot'], weight='bold')

        draw_compass(ax, self.wcs, nx, ny)

        ax.set_xlim(-0.5, nx - 0.5)
        ax.set_ylim(-0.5, ny - 0.5)
        ax.set_autoscale_on(False)
        fig.canvas.draw_idle()

    def update_display(self):
        if self.data is None:
            return
        img, norm = display_image(self.data, self.mode.value, self.pct.value)
        self.im.set_data(img)
        self.im.set_norm(norm)
        self.fig.canvas.draw_idle()

    def reset_points(self, redraw=False):
        for a in self.artists:
            a.remove()
        self.pts, self.artists, self.result = [], [], None
        if hasattr(self, 'txt'):
            self.txt['clkpa'].set_text('PA: click two points')
            self.txt['clk1'].set_text('')
            self.txt['clk2'].set_text('')
        if redraw:
            self.fig.canvas.draw_idle()

    # ---- interaction ----
    def on_move(self, ev):
        """Show pixel position, raw pixel value and RA/Dec under the cursor."""
        if self.data is None or ev.inaxes is not self.ax or ev.xdata is None:
            self.cursor.value = '&nbsp;'
            return
        ix, iy = int(round(ev.xdata)), int(round(ev.ydata))
        ny, nx = self.data.shape
        if not (0 <= ix < nx and 0 <= iy < ny):
            self.cursor.value = '&nbsp;'
            return
        s = f'x, y = {ev.xdata:6.2f}, {ev.ydata:6.2f}   pixel [{ix}, {iy}] = {self.data[iy, ix]:.4g} {self.unit}'
        c = pix2sky(self.wcs, ev.xdata, ev.ydata)
        if c is not None:
            s += f'   RA, Dec = {c.ra.deg:.6f}, {c.dec.deg:+.6f}'
        self.cursor.value = f'<span style="font-family:monospace; font-size:13px">{s}</span>'

    def on_click(self, ev):
        if (self.data is None or ev.inaxes is not self.ax or ev.button != 1
                or ev.xdata is None or toolbar_active(self.fig.canvas)):
            return
        if len(self.pts) >= 2:
            self.reset_points()

        x, y = ev.xdata, ev.ydata
        self.pts.append((x, y))

        c = Circle((x, y), CIRCLE_RADIUS_PIX, fill=False, ec='tab:red', lw=1.5)
        self.ax.add_patch(c)
        lab = self.ax.text(x + CIRCLE_RADIUS_PIX, y + CIRCLE_RADIUS_PIX, str(len(self.pts)),
                           color='tab:red', fontsize=FONT['annot'], weight='bold')
        self.artists += [c, lab]

        if len(self.pts) == 2:
            self.compute_pa()
        self.fig.canvas.draw_idle()

    def compute_pa(self):
        (x1, y1), (x2, y2) = self.pts
        ln, = self.ax.plot([x1, x2], [y1, y2], color='tab:red', lw=1.2)
        self.artists.append(ln)

        pa = pa_deg(x1, y1, x2, y2)
        pa_sky, sep_as, radec = sky_pa_sep(self.wcs, x1, y1, x2, y2)
        self.txt['clkpa'].set_text(f'PA = {pa:.1f}°  (CCW from +x)     sep = {sep_as:.2f}"')
        self.txt['clk1'].set_text(coord_str(1, x1, y1, radec[0], radec[1]))
        self.txt['clk2'].set_text(coord_str(2, x2, y2, radec[2], radec[3]))

        row = self.row
        self.result = {
            'name': row[COLS['name']],
            'ra': row[COLS['ra']], 'dec': row[COLS['dec']],
            'band': self.band,
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'ra1': radec[0], 'dec1': radec[1], 'ra2': radec[2], 'dec2': radec[3],
            'sep_arcsec': sep_as,
            'pa_pix_deg': pa,          # CCW from +x, [0, 180)
            'pa_sky_deg': pa_sky,      # east of north, [0, 180)
        }
        with self.out:
            self.out.clear_output()
            print(f'PA (CCW from +x) = {pa:.2f} deg   PA (E of N) = {pa_sky:.2f} deg   '
                  f'sep = {sep_as:.3f}"')

    # ---- output ----
    def save(self, _=None):
        if self.result is None:
            self.status.value = '<span style="color:red">Select two points before saving.</span>'
            return
        res = self.result
        png = os.path.join(OUT_DIR, f"{file_name(res['name'])}_{res['band']}.png")

        cols = list(res) if OUTPUT_COLUMNS is None else list(OUTPUT_COLUMNS)
        unknown = [k for k in cols if k not in res]
        if unknown:
            self.status.value = (f'<span style="color:red">Unknown OUTPUT_COLUMNS {unknown}; '
                                 f'available: {list(res)}</span>')
            return
        rec = {k: res[k] for k in cols}

        self.fig.savefig(png, dpi=150, bbox_inches='tight')

        df = (pd.read_csv(RESULT_CSV, dtype={'name': str}, keep_default_na=False, na_values=[''])
              if os.path.exists(RESULT_CSV) else pd.DataFrame())
        if len(df) and 'name' in df.columns and 'name' in rec:
            df = df[df['name'].astype(str) != str(rec['name'])]   # overwrite re-measured target
        df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)
        df.to_csv(RESULT_CSV, index=False, float_format=FLOAT_FORMAT)

        msg = f'Saved {png} and updated {RESULT_CSV}'
        if self.advance.value and self.dd.value < len(self.t) - 1:
            self.step(+1)
        self.status.value = msg

In [ ]:
# read everything as text so names keep their exact original string
targets = pd.read_csv(TARGET_CSV, encoding='utf-8-sig', skipinitialspace=True,
                      dtype=str, keep_default_na=False)
targets.columns = targets.columns.str.strip()

# match COLS to the CSV header (case-insensitive)
lower = {c.lower(): c for c in targets.columns}
for k, c in list(COLS.items()):
    if c in targets.columns:
        continue
    if c.lower() in lower:
        COLS[k] = lower[c.lower()]
    else:
        raise KeyError(f'No CSV column for {k!r} (looked for {c!r}). CSV columns are '
                       f'{list(targets.columns)}; edit COLS in the config cell.')

# numeric columns
for k in ('ra', 'dec'):
    targets[COLS[k]] = pd.to_numeric(targets[COLS[k]], errors='coerce')
bad = targets[COLS['ra']].isna() | targets[COLS['dec']].isna()
if bad.any():
    print(f'Dropping {bad.sum()} row(s) with unreadable RA/Dec:',
          list(targets.loc[bad, COLS['name']]))
    targets = targets[~bad]

vetter = PAVetter(targets)
display(vetter.ui)

In [ ]:
# Optional: inspect what the cutout service returns for the current target
# print(type(vetter.cutout), vetter.band)
# print(vetter.wcs)